<a href="https://colab.research.google.com/github/luciazhng-web/GeoLife_Trajectory_Mode_Detection/blob/main/1_RawTrajectoryExtraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Step1 - Trajectory Data Preparation**
**Draft data model:**

  user_id, trip_id, lat, lon, alt, datetime, modes, file_name(for debugging)

**Steps:**

1. Loop for all trajectories files in each "/user_id/trajectory" folders
2. For each .plt file, load the data by skipping the first 6 header lines, extract the first 2 columns(coordinates), 4th column(altitude), and combine 6th(date) and 7th(time) column into standard DateTime(timestamp)
3. Check for duplications(based on timestamp) per entries(keep a log file for any duplications)
- trajectory_log.txt: print "timestamp + current .plt file name + existing value in file_name column"
4. Store all trajectories temporary in "trajectory_df"
5. Load label.txt and merge the 1st column with the 2nd column as "timestamp_start" and 1st column with 3rd column as "timestamp_end" in standard DateTime format
6. Store all labels temporary in "label_df"
7. Match trajectory_df with label_df via condition "timestamp_start≤trajectory_df[datetime]≤timestamp_end" with modes, and assign a trip_id based on "user_id+5digits"
8. Repeat for all "/user_id" folders under raw data to get a full single dataset with name "raw_trajectory_labeled_df".

Setup environment

In [ ]:
import pandas as pd
import numpy as np
import os
import ast
import glob
import logging

import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from scipy.stats import mode
from os import preadv
from tqdm import tqdm
from pyproj import CRS, Transformer, Proj

# Mount Google Drive
drive.mount('/content/drive')
DATA_ROOT_DIR = '/content/drive/MyDrive/its_geolife/rawdata'

logging.basicConfig(
    filename='trajectory_log.txt',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger()
print(f"Set DATA_ROOT_DIR to: {DATA_ROOT_DIR}")

Load Trajectory data

In [ ]:
def get_trajectory(user_dir, user_id):

    trajectory_folder = os.path.join(user_dir, 'trajectory')
    all_user_trajectories = []

    plt_files = glob.glob(os.path.join(trajectory_folder, '*.plt'))

    if not plt_files:
        logger.warning(f"No .plt files found in {trajectory_folder}. Skipping user {user_id}.")
        return pd.DataFrame()

    for file_path in plt_files:
        file_name = os.path.basename(file_path)

        try:
            # Load the data by skipping the first 6 header lines
            df_part = pd.read_csv(
                file_path,
                skiprows=6,
                header=None,
                sep=',',
                encoding='gbk'
            )

            if df_part.shape[1] < 7:
                logger.error(f"File {file_name} has fewer than 7 columns. Skipping.")
                continue

            # Extract and rename columns
            df_part = df_part.iloc[:, [0, 1, 3, 5, 6]]
            df_part.columns = ['lat', 'lon', 'alt', 'date', 'time']

            # Combine date and time into standard DateTime(timestamp)
            df_part['datetime'] = pd.to_datetime(df_part['date'] + ' ' + df_part['time'], format='%Y/%m/%d %H:%M:%S')

            # Add metadata columns
            df_part['user_id'] = user_id
            df_part['file_name'] = file_name

            # Select final required columns and append
            df_part = df_part[['user_id', 'lat', 'lon', 'alt', 'datetime', 'file_name']]
            all_user_trajectories.append(df_part)

        except Exception as e:
            logger.error(f"Error processing file {file_name} for user {user_id}: {e}")
            continue

    if not all_user_trajectories:
        return pd.DataFrame()

    # Store all trajectories temporary in "trajectory_df"
    trajectory_df = pd.concat(all_user_trajectories, ignore_index=True)

    # Check for duplications (based on timestamp)
    duplicates = trajectory_df[trajectory_df.duplicated(subset=['datetime'], keep=False)]
    if not duplicates.empty:
        # Log duplicates
        for index, row in duplicates.iterrows():
            logger.info(
                f"DUPLICATE TIMESTAMP: {row['datetime']} found in file: {row['file_name']} (User: {row['user_id']})"
            )
        trajectory_df.drop_duplicates(subset=['datetime'], keep='first', inplace=True)
        logger.info(f"Removed {len(duplicates) - len(duplicates.drop_duplicates(subset=['datetime']))} duplicates for user {user_id}.")

    return trajectory_df.sort_values(by='datetime').reset_index(drop=True)

Load Label

In [ ]:
def get_labels(user_dir, user_id):

    labels_path = os.path.join(user_dir, 'labels.txt')

    if not os.path.exists(labels_path):
        logger.warning(f"labels.txt not found for user {user_id}. Skipping labels.")
        return pd.DataFrame()

    try:
        # Load label.txt - data is tab-separated, skip the first header line
        label_df = pd.read_csv(
            labels_path,
            skiprows=0,
            sep='\t',
            parse_dates=False
        )

        label_df.columns = ['Date', 'StartTime', 'EndTime', 'modes']

        # Merge columns as standard DateTime
        label_df['timestamp_start'] = pd.to_datetime(label_df['Date'] + ' ' + label_df['StartTime'], format='%Y/%m/%d %H:%M:%S')
        label_df['timestamp_end'] = pd.to_datetime(label_df['Date'] + ' ' + label_df['EndTime'], format='%Y/%m/%d %H:%M:%S')

        # Drop original date/time columns
        label_df = label_df.drop(columns=['Date', 'StartTime', 'EndTime'])

        # Add user_id and sequential index for Trip_ID
        label_df['user_id'] = user_id
        label_df['trip_sequence'] = label_df.index.astype(str).str.zfill(5)

        # Assign a trip_id based on "user_id+5digits"
        label_df['trip_id'] = label_df['user_id'] + '_' + label_df['trip_sequence']
        label_df = label_df.drop(columns=['trip_sequence'])

        # Store all labels temporary in "label_df"
        return label_df[['user_id', 'trip_id', 'timestamp_start', 'timestamp_end', 'modes']]

    except Exception as e:
        logger.error(f"Error processing labels.txt for user {user_id}: {e}")
        return pd.DataFrame()

Match trajectoroies with labels

In [ ]:
def label_match(trajectory_df, label_df, user_id):

    if trajectory_df.empty or label_df.empty:
        logger.info(f"Skipping merge for user {user_id} due to missing trajectory or label data.")
        trajectory_df['modes'] = pd.NA
        trajectory_df['trip_id'] = pd.NA
        return trajectory_df

    # Initialize 'modes' and 'trip_id' columns
    trajectory_df['modes'] = pd.NA
    trajectory_df['trip_id'] = pd.NA

    # Match trajectory_df with label_df via condition
    for index, row in label_df.iterrows():
        start = row['timestamp_start']
        end = row['timestamp_end']
        mode = row['modes']
        trip_id = row['trip_id']

        # Conditional assignment: start <= trajectory_df[datetime] <= end
        mask = (trajectory_df['datetime'] >= start) & (trajectory_df['datetime'] <= end)

        # Assign the label information to the filtered points
        trajectory_df.loc[mask, 'modes'] = mode
        trajectory_df.loc[mask, 'trip_id'] = trip_id

    return trajectory_df

Get the raw trajectory dataset with labels

In [ ]:
# Main Execution

if not os.path.isdir(DATA_ROOT_DIR):
    print(f"ERROR: Data root directory not found: {DATA_ROOT_DIR}")
    print("Please check the path and ensure Google Drive is correctly mounted.")
else:
    # Find all user folders (e.g., '006', '007')
    user_ids = [name for name in os.listdir(DATA_ROOT_DIR)
                if os.path.isdir(os.path.join(DATA_ROOT_DIR, name)) and name.isdigit()] # Filter to ensure only numeric IDs

    if not user_ids:
        print(f"Warning: No numeric user ID subdirectories found directly under {DATA_ROOT_DIR}.")
    else:
        print(f"Found {len(user_ids)} potential user directories.")

        all_labeled_trajectories = []

        # Repeat for all "/user_id" folders under raw data
        for user_id in sorted(user_ids):
            user_dir = os.path.join(DATA_ROOT_DIR, user_id)
            print(f"--- Processing User: {user_id} ---")

            # Load and process trajectories
            trajectory_df = get_trajectory(user_dir, user_id)
            if trajectory_df.empty:
                print(f"Skipping user {user_id}: No valid trajectory data found.")
                continue

            # Load and process labels
            label_df = get_labels(user_dir, user_id)

            # Integrate and label the points
            labeled_user_df = label_match(trajectory_df, label_df, user_id)

            all_labeled_trajectories.append(labeled_user_df)
            print(f"User {user_id} processed. {len(labeled_user_df)} total points.")

        # Combine all users into the final DataFrame
        if all_labeled_trajectories:
            raw_trajectory_labeled_df = pd.concat(all_labeled_trajectories, ignore_index=True)

            print("\n*** FINAL DATASET GENERATED ***")
            print(f"Total labeled and raw points: {len(raw_trajectory_labeled_df)}")
            print(f"Unique modes identified: {raw_trajectory_labeled_df['modes'].dropna().unique()}")
            print(f"Points with assigned mode (labeled trips): {raw_trajectory_labeled_df['modes'].count()}")

            # ----------------------------------------------------
            # Save the final result to CSV back into Google Drive
            # ----------------------------------------------------
            output_path = os.path.join(DATA_ROOT_DIR, 'raw_trajectory_labeled_df.csv')
            print(f"\nSaving final dataset to '{output_path}'")
            raw_trajectory_labeled_df.to_csv(output_path, index=False)

            # Display first 5 rows for verification
            print("\nFirst 5 rows of the combined DataFrame:")
            display(raw_trajectory_labeled_df.head())

        else:
            print("\n*** PROCESS FAILED ***")
            print("No data was successfully processed or concatenated.")

Export the raw trajectory file into the destination, end of 1_RawTrajectoryExtraction.